# 19 — Entraînement sur la cohorte élargie


## 1. Charger le dernier export du notebook 18

In [ ]:
from pathlib import Path
from datetime import datetime
import json
import platform
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path(r"E:\Projet_Alzheimer")
TABLES = ROOT / "data" / "02_Tables_originales"
MANIFESTES = ROOT / "data" / "05_Manifestes"

dossiers = sorted(MANIFESTES.glob("extension_tabulaire_18_*"))
if not dossiers:
    raise FileNotFoundError("Aucun export du notebook 18 : lance-le d'abord.")
EXPORT18 = dossiers[-1]

cohorte = pd.read_csv(EXPORT18 / "cohorte_tabulaire_elargie.csv", encoding="utf-8-sig")
cohorte["date_T0"] = pd.to_datetime(cohorte["date_T0"], errors="coerce")
protocole18 = json.loads((EXPORT18 / "protocole.json").read_text(encoding="utf-8"))

print("Python :", sys.executable)
print("Export 18 utilise :", EXPORT18.name)
print("Participants :", len(cohorte))
print("Cible 1 (s'aggrave d'ici 2 ans) :", int(cohorte["cible_aggravation_2ans"].sum()),
      f"({cohorte['cible_aggravation_2ans'].mean():.1%})")
print("Horizon :", protocole18["horizon_jours"], "jours, fenetre", protocole18["fenetre_jours"])

display(pd.crosstab(cohorte["dx_T0"].map({1: "CN", 2: "MCI"}),
                    cohorte["cible_aggravation_2ans"], margins=True))

## 2. Récupérer l'âge, absent à 100 %

Dans l'export du notebook 18, `age_reference` est manquant pour **tous** les participants,
alors que le sexe et le niveau d'études sortent de la même table. Deux causes :

1. l'âge y est cherché avec une tolérance de 365 jours avant T0 ; si la ligne de
   démographie qui porte l'année de naissance est plus ancienne, elle est perdue ;
2. `PTDOBYY` n'est pas toujours un nombre pur, et `pd.to_numeric` le rejette.

Les notebooks 07 et 07c faisaient autrement : **première valeur non vide du participant,
sans fenêtre de date**, puis extraction des 4 chiffres de l'année. C'est légitime :
une année de naissance ne change pas avec le temps, elle ne peut donc pas venir du futur.

On applique la même méthode ici. Si elle échoue aussi, l'âge est simplement écarté et
le modèle tourne sans lui — ce sera écrit noir sur blanc.

In [ ]:
demo_path = sorted(TABLES.rglob("All_Subjects_PTDEMOG_*.csv"))
age_recupere = False

if not demo_path:
    print("Table PTDEMOG introuvable : on continue sans l'age.")
else:
    demo = pd.read_csv(demo_path[-1], dtype="string", encoding="utf-8-sig")
    demo.columns = demo.columns.str.strip().str.upper()
    demo["PTID"] = demo["PTID"].str.strip().str.upper()

    colonne_dob = next((c for c in ("PTDOBYY", "PTDOB", "PTDOBYEAR") if c in demo.columns), None)
    print("Colonne de naissance utilisee :", colonne_dob)

    if colonne_dob is not None:
        d = demo.loc[demo["PTID"].isin(set(cohorte["PTID"])), ["PTID", colonne_dob]].copy()
        d[colonne_dob] = d[colonne_dob].astype("string").str.strip()
        d = d.loc[d[colonne_dob].notna() & d[colonne_dob].ne("")]
        # premiere valeur non vide par participant, sans fenetre de date
        d = d.drop_duplicates("PTID", keep="first")
        annee = d[colonne_dob].str.extract(r"(\d{4})")[0]
        d["annee_naissance_recuperee"] = pd.to_numeric(annee, errors="coerce")
        d = d.loc[d["annee_naissance_recuperee"].between(1850, 2026),
                  ["PTID", "annee_naissance_recuperee"]]

        cohorte = cohorte.merge(d, on="PTID", how="left", validate="one_to_one")
        cohorte["age_reference"] = (cohorte["date_T0"].dt.year
                                    - cohorte["annee_naissance_recuperee"])
        cohorte.loc[~cohorte["age_reference"].between(40, 120), "age_reference"] = np.nan
        n = int(cohorte["age_reference"].notna().sum())
        age_recupere = n > 0
        print(f"Age recupere pour {n} / {len(cohorte)} participants "
              f"({n / len(cohorte):.1%})")
        if age_recupere:
            print("Age min / median / max :",
                  cohorte["age_reference"].min(),
                  cohorte["age_reference"].median(),
                  cohorte["age_reference"].max())

if not age_recupere:
    print("!! L'age reste indisponible : il sera exclu des variables.")

## 3. Variables explicatives et contrôle anti-fuite

Deux décisions à expliquer en soutenance.

**On ajoute `dx_T0`** (l'état CN ou MCI au départ). Dans le notebook 17 cette variable
ne changeait rien, parce que la cohorte était presque entièrement MCI. Ici la cohorte
mélange 490 CN et 698 MCI, et les deux groupes n'ont pas du tout le même risque :
7 % contre 25 %. C'est une information connue à T0, donc parfaitement autorisée,
et il serait absurde de la cacher au modèle.

**On exclut** toutes les colonnes d'audit : les dates, le diagnostic de la visite cible,
le délai jusqu'à cette visite, la durée de suivi observée, les réversions, le nombre de
variables présentes. Elles décrivent ce qui s'est passé **après** T0.

Le même garde-fou que dans le notebook 17 vérifie ensuite qu'aucune variable restante
n'est anormalement corrélée à la réponse.

In [ ]:
CIBLE = "cible_aggravation_2ans"

FUITES = {
    "dx_cible", "date_cible", "delai_jours", "suivi_observe_jours",
    "reversion_a_la_visite_cible", CIBLE,
}
EXCLURE_EXACT = {
    "PTID", "statut", "nb_variables_presentes", "exploitable_tabulaire",
    "annee_naissance", "annee_naissance_recuperee", "present_ancienne_cohorte",
} | FUITES
EXCLURE_PREFIXE = ("date_", "anciennete_", "Unnamed")

variables = [c for c in cohorte.columns
             if c not in EXCLURE_EXACT
             and not c.startswith(EXCLURE_PREFIXE)
             and pd.api.types.is_numeric_dtype(cohorte[c])]
if not age_recupere and "age_reference" in variables:
    variables.remove("age_reference")
variables = [c for c in variables if cohorte[c].notna().any()]

X = cohorte[variables].to_numpy(dtype=float)
y = cohorte[CIBLE].to_numpy(dtype=int)

print(f"{len(variables)} variables explicatives :")
print(variables)
print()
manquants = (cohorte[variables].isna().mean() * 100).round(1).sort_values(ascending=False)
print("Taux de valeurs manquantes (%) :")
display(manquants.to_frame("manquants_%"))

# --- garde-fou anti-fuite (identique au notebook 17) ---
corr = (cohorte[variables]
        .apply(lambda s: s.corr(pd.Series(y, index=s.index)))
        .abs().sort_values(ascending=False))
print("Correlation absolue avec la cible (5 plus fortes) :")
display(corr.head(5).to_frame("correlation"))
if corr.max() > 0.95:
    print("!! ALERTE : correlation > 0.95 — une colonne-cible est restee dans les variables.")
elif corr.max() > 0.80:
    print("!  Vigilance : correlation > 0.80, verifie le sens de cette variable.")
else:
    print("-> OK, aucune variable suspecte.")

## 4. Protocole d'évaluation

Strictement le même que dans les notebooks 11b et 17, pour que les comparaisons aient
un sens : `StratifiedKFold(5, shuffle=True, random_state=0)`, imputation par la médiane
**apprise dans le pli d'entraînement**, mise à l'échelle pour la régression logistique,
`class_weight="balanced"` pour compenser le déséquilibre (17,8 % de positifs).

Trois modèles : régression logistique, random forest, et leur ensemble
(la moyenne de leurs probabilités), comme le modèle final du notebook 17.

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

def tuyau_logreg():
    return Pipeline([("imp", SimpleImputer(strategy="median")),
                     ("ech", StandardScaler()),
                     ("clf", LogisticRegression(max_iter=5000, class_weight="balanced"))])

def tuyau_foret():
    return Pipeline([("imp", SimpleImputer(strategy="median")),
                     ("clf", RandomForestClassifier(n_estimators=300, random_state=0,
                                                    class_weight="balanced", n_jobs=-1))])

NOMS = ["Regression logistique", "Random forest", "Ensemble (moyenne des deux)"]

def evaluer(X, y, cv=CV):
    """Retourne (auc par pli, probabilites hors echantillon) pour les 3 modeles."""
    auc = {n: [] for n in NOMS}
    oof = {n: np.full(len(y), np.nan) for n in NOMS}
    for app, test in cv.split(X, y):
        p = {}
        p[NOMS[0]] = tuyau_logreg().fit(X[app], y[app]).predict_proba(X[test])[:, 1]
        p[NOMS[1]] = tuyau_foret().fit(X[app], y[app]).predict_proba(X[test])[:, 1]
        p[NOMS[2]] = (p[NOMS[0]] + p[NOMS[1]]) / 2
        for n in NOMS:
            auc[n].append(roc_auc_score(y[test], p[n]))
            oof[n][test] = p[n]
    return auc, oof

auc_par_pli, oof = evaluer(X, y)

classement = (pd.DataFrame({
    "modele": NOMS,
    "roc_auc_moyenne": [np.mean(auc_par_pli[n]) for n in NOMS],
    "roc_auc_ecart_type": [np.std(auc_par_pli[n]) for n in NOMS],
}).sort_values("roc_auc_moyenne", ascending=False).reset_index(drop=True))

print(f"=== Cohorte elargie : {len(y)} participants, {y.mean():.1%} de positifs ===")
display(classement.round(3))
display(pd.DataFrame(auc_par_pli, index=[f"pli {i}" for i in range(1, 6)]).round(3))

MEILLEUR = classement.loc[0, "modele"]
print("Meilleur modele :", MEILLEUR)

## 5. Est-ce que plus de participants améliore vraiment ?

C'est la question qui motive toute cette extension, et on peut y répondre proprement.

On met de côté **un jeu de validation fixe** (25 % des participants, jamais utilisé pour
entraîner). On entraîne ensuite le même modèle sur des échantillons de taille croissante
— 100, 200, 312, 450, 600, 750, puis tout le reste — avec plusieurs tirages au hasard
pour chaque taille, et on mesure la note sur ce jeu de validation **toujours identique**.

Lecture de la courbe :
- si elle **monte encore** à droite, davantage de participants aiderait ;
- si elle est **plate**, le plafond ne vient pas du nombre de participants.

Le point à **312** est volontaire : c'est la taille de la cohorte du notebook 17.

In [ ]:
from sklearn.model_selection import train_test_split

X_pool, X_val, y_pool, y_val = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=0)
print(f"Validation fixe : {len(y_val)} participants ({y_val.mean():.1%} de positifs)")
print(f"Reservoir d'entrainement : {len(y_pool)} participants")

TAILLES = [t for t in [100, 200, 312, 450, 600, 750, len(y_pool)] if t <= len(y_pool)]
TAILLES = sorted(set(TAILLES))
TIRAGES = 5

def modele_ensemble(Xa, ya, Xt):
    pl = tuyau_logreg().fit(Xa, ya).predict_proba(Xt)[:, 1]
    pf = tuyau_foret().fit(Xa, ya).predict_proba(Xt)[:, 1]
    return (pl + pf) / 2

lignes = []
for taille in TAILLES:
    scores = []
    for graine in range(TIRAGES):
        if taille == len(y_pool):
            Xa, ya = X_pool, y_pool
        else:
            Xa, _, ya, _ = train_test_split(
                X_pool, y_pool, train_size=taille, stratify=y_pool, random_state=graine)
        scores.append(roc_auc_score(y_val, modele_ensemble(Xa, ya, X_val)))
        if taille == len(y_pool):
            break
    lignes.append({"n_entrainement": taille,
                   "roc_auc_moyenne": float(np.mean(scores)),
                   "roc_auc_ecart_type": float(np.std(scores)),
                   "tirages": len(scores)})
    print(f"  n = {taille:4d} -> AUC {np.mean(scores):.3f} (+/- {np.std(scores):.3f})")

courbe = pd.DataFrame(lignes)
display(courbe.round(3))

gain = courbe.iloc[-1]["roc_auc_moyenne"] - courbe.loc[
    courbe["n_entrainement"].sub(312).abs().idxmin(), "roc_auc_moyenne"]
print(f"\nEcart entre la taille du notebook 17 (~312) et la cohorte complete : {gain:+.3f}")

fig, ax = plt.subplots(figsize=(7.5, 4.5), constrained_layout=True)
ax.errorbar(courbe["n_entrainement"], courbe["roc_auc_moyenne"],
            yerr=courbe["roc_auc_ecart_type"], marker="o", capsize=4, color="#4C72B0")
ax.axhline(0.5, color="grey", ls="--", lw=1, label="hasard")
ax.axvline(312, color="#C44E52", ls=":", lw=1.5, label="taille du notebook 17")
ax.set_xlabel("Nombre de participants d'entrainement")
ax.set_ylabel("ROC-AUC sur la validation fixe")
ax.set_title("Est-ce que plus de participants ameliore la prediction ?")
ax.legend()
plt.show()

## 6. Un chiffre réellement comparable au notebook 17

Les sections précédentes utilisent la nouvelle cible (deux ans). Pour comparer au 0,813
du notebook 17, il faut la **même définition de cible** : « le diagnostic s'aggrave à
n'importe quel moment après T0 ».

On la reconstruit ici sur la même cohorte élargie, à partir de la table des diagnostics.
Les variables, elles, ne changent pas : toujours mesurées au plus tard à T0.

Attention en soutenance : même avec la même définition de cible, T0 n'est pas le même
(premier diagnostic ici, première IRM dans le notebook 17) et la population non plus.
C'est une comparaison *indicative*, pas une égalité.

In [ ]:
dx_files = sorted(TABLES.rglob("All_Subjects_DXSUM_*.csv"))
comparaison_cibles = pd.DataFrame()

if not dx_files:
    print("Table DXSUM introuvable : section ignoree.")
else:
    dxb = pd.read_csv(dx_files[-1], dtype="string", encoding="utf-8-sig")
    dxb.columns = dxb.columns.str.strip().str.upper()
    dxb["PTID"] = dxb["PTID"].str.strip().str.upper()
    dxb["date_dx"] = pd.to_datetime(dxb["EXAMDATE"].astype("string").str.strip().str.slice(0, 10),
                                    format="%Y-%m-%d", errors="coerce").dt.normalize()
    dxb["code_dx"] = pd.to_numeric(dxb["DIAGNOSIS"], errors="coerce")
    dxb = dxb.loc[dxb["PTID"].isin(set(cohorte["PTID"]))
                  & dxb["date_dx"].notna() & dxb["code_dx"].isin([1, 2, 3]),
                  ["PTID", "date_dx", "code_dx"]]

    suite = dxb.merge(cohorte[["PTID", "date_T0", "dx_T0"]], on="PTID", validate="many_to_one")
    suite = suite.loc[suite["date_dx"] > suite["date_T0"]]
    futur = suite.groupby("PTID")["code_dx"].max().rename("dx_futur_max")

    tmp = cohorte[["PTID", "dx_T0", CIBLE]].merge(futur, on="PTID", how="left")
    tmp["cible_tout_moment"] = (tmp["dx_futur_max"] > tmp["dx_T0"]).astype("Int64")
    tmp.loc[tmp["dx_futur_max"].isna(), "cible_tout_moment"] = pd.NA

    utilisable = tmp["cible_tout_moment"].notna().to_numpy()
    y_ancienne = tmp.loc[utilisable, "cible_tout_moment"].to_numpy(dtype=int)
    X_ancienne = X[utilisable]

    print(f"Participants avec au moins une visite apres T0 : {utilisable.sum()}")
    print(f"Cible « aggravation a tout moment » : {y_ancienne.sum()} positifs "
          f"({y_ancienne.mean():.1%})")
    print(f"Cible « a deux ans »                : {int(tmp[CIBLE].sum())} positifs "
          f"({tmp[CIBLE].mean():.1%})\n")

    display(pd.crosstab(tmp[CIBLE], tmp["cible_tout_moment"],
                        rownames=["a 2 ans"], colnames=["a tout moment"], margins=True))

    auc_anc, _ = evaluer(X_ancienne, y_ancienne)
    comparaison_cibles = pd.DataFrame({
        "modele": NOMS,
        "AUC cible a 2 ans": [np.mean(auc_par_pli[n]) for n in NOMS],
        "AUC cible a tout moment": [np.mean(auc_anc[n]) for n in NOMS],
    }).round(3)
    print("\n=== Les deux cibles, memes variables, meme cohorte ===")
    display(comparaison_cibles)
    print("Rappel notebook 17 (cible a tout moment, 312 participants, T0 = premiere IRM) : 0.813")

## 7. Seuil d'alerte, matrice de confusion et calibration

Même démarche que dans le notebook 17. Le modèle produit une probabilité ; le seuil
décide à partir de quelle valeur on alerte. Avec seulement 17,8 % de positifs, le seuil
par défaut de 0,50 sera encore moins adapté qu'avant.

In [ ]:
from sklearn.metrics import roc_curve, confusion_matrix, balanced_accuracy_score
from sklearn.calibration import calibration_curve

proba = oof[MEILLEUR]
fpr, tpr, seuils = roc_curve(y, proba)
i_youden = int(np.argmax(tpr - fpr))
seuil_youden = float(seuils[i_youden])
ok80 = np.where(tpr >= 0.80)[0]
seuil_sens80 = float(seuils[ok80[0]]) if len(ok80) else seuil_youden

def mesures(s):
    pred = (proba >= s).astype(int)
    vn, fp, fn, vp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return {"seuil": round(s, 3),
            "sensibilite": round(vp / max(vp + fn, 1), 3),
            "specificite": round(vn / max(vn + fp, 1), 3),
            "VPP": round(vp / max(vp + fp, 1), 3),
            "VPN": round(vn / max(vn + fn, 1), 3),
            "exactitude equilibree": round(balanced_accuracy_score(y, pred), 3)}

print("Modele evalue :", MEILLEUR)
print(f"Reference naive (toujours « ne s'aggrave pas ») : exactitude = {1 - y.mean():.3f}, "
      f"exactitude equilibree = 0.500\n")

table_seuils = pd.DataFrame([
    {"strategie": "0,50 (par defaut)", **mesures(0.50)},
    {"strategie": "Youden (equilibre)", **mesures(seuil_youden)},
    {"strategie": "sensibilite >= 80 %", **mesures(seuil_sens80)},
])
display(table_seuils)

pred = (proba >= seuil_youden).astype(int)
mat = pd.DataFrame(confusion_matrix(y, pred, labels=[0, 1]),
                   index=["reel : ne s'aggrave pas", "reel : s'aggrave"],
                   columns=["predit : ne s'aggrave pas", "predit : s'aggrave"])
print(f"\n=== Matrice de confusion au seuil de Youden ({seuil_youden:.3f}) ===")
display(mat)

fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4.2), constrained_layout=True)
a1.plot(fpr, tpr, color="#4C72B0", lw=2,
        label=f"AUC = {classement.loc[0, 'roc_auc_moyenne']:.3f}")
a1.plot([0, 1], [0, 1], color="grey", ls="--", lw=1, label="hasard")
a1.scatter([fpr[i_youden]], [tpr[i_youden]], color="#C44E52", s=70, zorder=5)
a1.annotate(f"seuil {seuil_youden:.2f}", (fpr[i_youden], tpr[i_youden]),
            textcoords="offset points", xytext=(8, -12), fontsize=9)
a1.set_xlabel("1 - specificite"); a1.set_ylabel("sensibilite")
a1.set_title("Courbe ROC — cohorte elargie", fontsize=10); a1.legend(loc="lower right")

frac, moy = calibration_curve(y, proba, n_bins=5, strategy="quantile")
a2.plot(moy, frac, marker="o", color="#4C72B0")
a2.plot([0, 1], [0, 1], color="grey", ls="--", lw=1)
a2.set_xlabel("probabilite predite"); a2.set_ylabel("frequence observee")
a2.set_title("Calibration (5 groupes)", fontsize=10)
plt.show()

## 8. Deux populations très différentes : CN et MCI

La cohorte mélange des participants normaux et des troubles légers. Leur risque n'a rien
à voir : environ 7 % contre 25 % d'aggravation à deux ans. Un modèle qui ne ferait que
distinguer CN de MCI obtiendrait déjà une note correcte — d'où l'intérêt de regarder
la performance **à l'intérieur** de chaque groupe.

Si la note reste bonne dans chaque sous-groupe, le modèle apporte vraiment une
information au-delà de l'état de départ.

In [ ]:
groupes = []
for code, nom in [(1, "CN au depart"), (2, "MCI au depart")]:
    m = (cohorte["dx_T0"] == code).to_numpy()
    if m.sum() < 30 or len(np.unique(y[m])) < 2:
        continue
    groupes.append({
        "groupe": nom,
        "participants": int(m.sum()),
        "positifs": int(y[m].sum()),
        "taux_positifs": round(float(y[m].mean()), 3),
        "AUC du modele dans le groupe": round(roc_auc_score(y[m], proba[m]), 3),
    })
sous_groupes = pd.DataFrame(groupes)
display(sous_groupes)

auc_dx_seul = roc_auc_score(y, cohorte["dx_T0"].to_numpy(dtype=float))
print(f"AUC en n'utilisant que l'etat de depart CN/MCI : {auc_dx_seul:.3f}")
print(f"AUC du modele complet                         : {classement.loc[0, 'roc_auc_moyenne']:.3f}")
print(f"Apport des autres variables                   : {classement.loc[0, 'roc_auc_moyenne'] - auc_dx_seul:+.3f}")

## 9. Export des résultats

In [ ]:
SORTIE = MANIFESTES / ("entrainement_19_" + datetime.now().strftime("%Y%m%d_%H%M%S_%f"))
SORTIE.mkdir(parents=True, exist_ok=False)

def sauver(df, nom):
    df.to_csv(SORTIE / nom, index=False, encoding="utf-8-sig")

sauver(classement, "classement_modeles.csv")
sauver(pd.DataFrame(auc_par_pli), "auc_par_pli.csv")
sauver(manquants.to_frame("manquants_%").reset_index().rename(columns={"index": "variable"}),
       "couverture_variables.csv")
sauver(courbe, "courbe_apprentissage.csv")
sauver(table_seuils, "table_seuils.csv")
sauver(mat.reset_index().rename(columns={"index": "reel"}), "matrice_confusion.csv")
if not sous_groupes.empty:
    sauver(sous_groupes, "sous_groupes_CN_MCI.csv")
if not comparaison_cibles.empty:
    sauver(comparaison_cibles, "comparaison_des_deux_cibles.csv")

resume = {
    "experience": "19_entrainement_cohorte_elargie",
    "date": datetime.now().isoformat(),
    "export_18": EXPORT18.name,
    "participants": int(len(y)),
    "positifs": int(y.sum()),
    "taux_positifs": round(float(y.mean()), 3),
    "variables": variables,
    "age_recupere": bool(age_recupere),
    "meilleur_modele": MEILLEUR,
    "roc_auc": round(float(classement.loc[0, "roc_auc_moyenne"]), 3),
    "roc_auc_ecart_type": round(float(classement.loc[0, "roc_auc_ecart_type"]), 3),
    "seuil_youden": round(seuil_youden, 3),
    "cible": protocole18["cible"],
    "comparabilite_notebook_17": "cible differente ; voir la section 6 pour un chiffre comparable",
    "python": platform.python_version(), "pandas": pd.__version__, "numpy": np.__version__,
}
(SORTIE / "resume.json").write_text(json.dumps(resume, ensure_ascii=False, indent=2),
                                    encoding="utf-8")

print("=" * 78)
print(f">>> COHORTE ELARGIE : {len(y)} participants ({y.mean():.1%} de positifs)")
print(f"    Meilleur modele : {MEILLEUR}")
print(f"    ROC-AUC = {classement.loc[0, 'roc_auc_moyenne']:.3f} "
      f"+/- {classement.loc[0, 'roc_auc_ecart_type']:.3f}")
print(f"    Seuil conseille (Youden) = {seuil_youden:.3f} -> sensibilite "
      f"{mesures(seuil_youden)['sensibilite']:.2f}, specificite "
      f"{mesures(seuil_youden)['specificite']:.2f}")
print(f"    Courbe d'apprentissage : {courbe.iloc[0]['roc_auc_moyenne']:.3f} a n="
      f"{int(courbe.iloc[0]['n_entrainement'])}  ->  "
      f"{courbe.iloc[-1]['roc_auc_moyenne']:.3f} a n={int(courbe.iloc[-1]['n_entrainement'])}")
print("=" * 78)
print("\nResultats enregistres dans :", SORTIE)